# EuroSAT multispectral image denoising with DnCNN

This notebook is the executable source of truth for the project. It compares two residual DnCNN strategies on 13-band Sentinel-2 tiles:

- **Model A** learns spatial and cross-band structure from all 13 bands jointly.
- **Model B** learns one shared single-band denoiser and applies it independently to every band.

The workflow covers configuration, a stratified 70/10/20 split, synthetic Gaussian corruption, training or checkpoint loading, PSNR/SSIM evaluation, paired statistical tests, and publication-ready plots.


## 1. Optional dependency installation

Create an environment from `requirements.txt` before opening the notebook. The commented command below is useful only when working in a fresh notebook kernel.


In [ ]:
# %pip install -r requirements.txt


## 2. Imports and configuration


In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
from dotenv import load_dotenv
from scipy import stats
from skimage.metrics import structural_similarity as ssim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

load_dotenv()

# Paths can be overridden in .env; relative paths resolve from the notebook's working directory.
PROJECT_ROOT = Path.cwd()
DATASET_PATH = Path(os.getenv("DATASET_PATH", PROJECT_ROOT / "data" / "EuroSATallBands")).expanduser()
MODEL_DIR = Path(os.getenv("MODEL_DIR", PROJECT_ROOT / "models")).expanduser()
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", PROJECT_ROOT / "outputs")).expanduser()
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42

# Training configuration
BATCH_SIZE = 128
NUM_WORKERS = 2
MAX_EPOCHS = 150
PATIENCE = 10
LEARNING_RATE = 1e-3
VALIDATION_SIGMA = 25

# Evaluation configuration
EVAL_SIGMAS = [15, 25, 50]
SWEEP_SIGMAS = list(range(5, 76, 5))
PER_CHANNEL_SAMPLES = 1000

# Execution switches. Supplied weights make evaluation the safe default.
TRAIN_MODEL_A = False
TRAIN_MODEL_B = False
RUN_DETAILED_EVALUATION = False

# True reproduces the supplied checkpoints/results (27,597 tiles). Set False to use
# only class-prefixed EuroSAT files (27,000 tiles) and exclude supplemental water scenes.
INCLUDE_SUPPLEMENTAL_WATER_SCENES = True
USE_DATA_PARALLEL = torch.cuda.device_count() > 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"CUDA devices: {torch.cuda.device_count()}")


## 3. Reproducibility helpers


In [ ]:
def set_seed(seed: int = SEED) -> None:
    """Set random seeds for repeatable splits and noise generation."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


## 4. Data loading and preprocessing


In [ ]:
def load_image_paths(
    root_dir: Path,
    include_supplemental: bool = INCLUDE_SUPPLEMENTAL_WATER_SCENES,
) -> Tuple[List[Path], List[int], List[str]]:
    """Return TIFF paths, integer labels, and class names from a class-folder dataset."""
    if not root_dir.is_dir():
        raise FileNotFoundError(
            f"Dataset directory not found: {root_dir}. "
            "Copy .env.example to .env and set DATASET_PATH."
        )

    image_paths: List[Path] = []
    labels: List[int] = []
    classes = sorted(path.name for path in root_dir.iterdir() if path.is_dir())

    for label, class_name in enumerate(classes):
        class_folder = root_dir / class_name
        candidates = sorted(class_folder.glob("*.tif"))
        if not include_supplemental:
            candidates = [path for path in candidates if path.name.startswith(f"{class_name}_")]
        image_paths.extend(candidates)
        labels.extend([label] * len(candidates))

    if not image_paths:
        raise FileNotFoundError(f"No .tif files found beneath {root_dir}")

    return image_paths, labels, classes


def load_image(path: Path) -> np.ndarray:
    """Load one 13-band tile and scale reflectance values to the [0, 1] range."""
    with rasterio.open(path) as src:
        if src.count != 13:
            raise ValueError(f"Expected 13 bands in {path}, found {src.count}")
        image = src.read().astype(np.float32)
    image = image / 10000.0
    return np.clip(image, 0.0, 1.0)


def add_noise(
    image: np.ndarray,
    sigma_min: float = 10,
    sigma_max: float = 55,
    fixed_sigma: Optional[float] = None,
) -> Tuple[np.ndarray, float]:
    """Add Gaussian noise with sigma expressed on the conventional 0-255 scale."""
    sigma = fixed_sigma if fixed_sigma is not None else np.random.uniform(sigma_min, sigma_max)
    noise = np.random.normal(0, sigma / 255.0, image.shape).astype(np.float32)
    noisy = np.clip(image + noise, 0.0, 1.0)
    return noisy, float(sigma)


def split_dataset(
    image_paths: Sequence[Path],
    labels: Sequence[int],
    val_size: float = 0.1,
    test_size: float = 0.2,
    seed: int = SEED,
) -> Tuple[List[Path], List[Path], List[Path], List[int], List[int], List[int]]:
    """Create reproducible, class-stratified train/validation/test splits."""
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        list(image_paths),
        list(labels),
        test_size=test_size,
        stratify=labels,
        random_state=seed,
    )

    adjusted_val_size = val_size / (1.0 - test_size)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_paths,
        train_labels,
        test_size=adjusted_val_size,
        stratify=train_labels,
        random_state=seed,
    )

    return train_paths, val_paths, test_paths, train_labels, val_labels, test_labels


## 5. Dataset classes


In [ ]:
class ModelADataset(Dataset):
    """Dataset for Model A: all 13 channels are denoised jointly."""

    def __init__(self, image_paths: Sequence[Path], labels: Sequence[int], fixed_sigma: Optional[float] = None):
        self.image_paths = list(image_paths)
        self.labels = list(labels)
        self.fixed_sigma = fixed_sigma

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        clean = load_image(self.image_paths[idx])
        noisy, _ = add_noise(clean, fixed_sigma=self.fixed_sigma)
        return torch.from_numpy(noisy), torch.from_numpy(clean), self.labels[idx]


class ModelBDataset(Dataset):
    """Dataset for Model B: each image contributes 13 single-channel training examples."""

    def __init__(self, image_paths: Sequence[Path], labels: Sequence[int], fixed_sigma: Optional[float] = None):
        self.image_paths = list(image_paths)
        self.labels = list(labels)
        self.fixed_sigma = fixed_sigma
        self.num_channels = 13

    def __len__(self) -> int:
        return len(self.image_paths) * self.num_channels

    def __getitem__(self, idx: int):
        image_idx = idx // self.num_channels
        channel_idx = idx % self.num_channels

        clean = load_image(self.image_paths[image_idx])
        clean_channel = clean[channel_idx : channel_idx + 1]
        noisy_channel, _ = add_noise(clean_channel, fixed_sigma=self.fixed_sigma)

        return torch.from_numpy(noisy_channel), torch.from_numpy(clean_channel), self.labels[image_idx]


## 6. Model and checkpoint helpers


In [ ]:
def build_dncnn(num_channels: int, depth: int = 17, features: int = 64) -> nn.Module:
    """Build a DnCNN-style residual noise predictor."""
    layers: List[nn.Module] = [
        nn.Conv2d(num_channels, features, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
    ]

    for _ in range(depth - 2):
        layers.extend([
            nn.Conv2d(features, features, kernel_size=3, padding=1),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
        ])

    layers.append(nn.Conv2d(features, num_channels, kernel_size=3, padding=1))
    return nn.Sequential(*layers)


def maybe_parallel(model: nn.Module) -> nn.Module:
    if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
        return nn.DataParallel(model)
    return model


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def model_slug(model_name: str) -> str:
    try:
        return {"ModelA": "model_a", "ModelB": "model_b"}[model_name]
    except KeyError as error:
        raise ValueError(f"Unknown model name: {model_name}") from error


def model_path(model_name: str) -> Path:
    return MODEL_DIR / f"best_{model_slug(model_name)}.pth"


def checkpoint_path(model_name: str) -> Path:
    return MODEL_DIR / f"checkpoint_{model_slug(model_name)}.pth"


def save_model_weights(model: nn.Module, path: Path) -> None:
    torch.save(unwrap_model(model).state_dict(), path)


def load_model_weights(model: nn.Module, path: Path, device: torch.device = DEVICE) -> nn.Module:
    """Load weights saved from either a plain model or a DataParallel model."""
    state_dict = torch.load(path, map_location=device, weights_only=True)
    cleaned = {key.replace("module.", "", 1): value for key, value in state_dict.items()}
    unwrap_model(model).load_state_dict(cleaned)
    return model


## 7. Metrics


In [ ]:
def psnr_torch(clean: torch.Tensor, denoised: torch.Tensor) -> float:
    mse = torch.mean((clean - denoised) ** 2)
    if mse.item() == 0:
        return float("inf")
    return float(20.0 * torch.log10(torch.tensor(1.0, device=clean.device) / torch.sqrt(mse)).item())


def ssim_torch(clean: torch.Tensor, denoised: torch.Tensor) -> float:
    clean_np = clean.detach().cpu().numpy()
    denoised_np = denoised.detach().cpu().numpy()

    scores = []
    for i in range(clean_np.shape[0]):
        for c in range(clean_np.shape[1]):
            scores.append(ssim(clean_np[i, c], denoised_np[i, c], data_range=1.0))
    return float(np.mean(scores))


def psnr_numpy(clean: np.ndarray, denoised: np.ndarray) -> float:
    mse = np.mean((clean - denoised) ** 2)
    if mse == 0:
        return float("inf")
    return float(20.0 * np.log10(1.0 / np.sqrt(mse)))


def ssim_numpy(clean: np.ndarray, denoised: np.ndarray) -> float:
    scores = [ssim(clean[c], denoised[c], data_range=1.0) for c in range(clean.shape[0])]
    return float(np.mean(scores))


## 8. Training and standard evaluation


In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    model_name: str,
    max_epochs: int = MAX_EPOCHS,
    patience: int = PATIENCE,
    lr: float = LEARNING_RATE,
) -> nn.Module:
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=5)
    criterion = nn.MSELoss()

    best_psnr = -float("inf")
    patience_counter = 0
    start_epoch = 0

    best_path = model_path(model_name)
    ckpt_path = checkpoint_path(model_name)

    if ckpt_path.exists():
        print(f"Resuming {model_name} from {ckpt_path}")
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)

        # Supports checkpoints from this cleaned notebook and from the original notebook.
        model_state = checkpoint.get("model_state", checkpoint.get("modelState"))
        optimizer_state = checkpoint.get("optimizer_state", checkpoint.get("optimizerState"))
        scheduler_state = checkpoint.get("scheduler_state", checkpoint.get("schedulerState"))
        best_psnr = checkpoint.get("best_psnr", checkpoint.get("bestPSNR", best_psnr))
        patience_counter = checkpoint.get("patience_counter", checkpoint.get("patienceCounter", patience_counter))

        cleaned_state = {key.replace("module.", "", 1): value for key, value in model_state.items()}
        unwrap_model(model).load_state_dict(cleaned_state)
        if optimizer_state is not None:
            optimizer.load_state_dict(optimizer_state)
        if scheduler_state is not None:
            scheduler.load_state_dict(scheduler_state)
        start_epoch = checkpoint["epoch"] + 1
        print(f"Resumed at epoch {start_epoch}; best validation PSNR = {best_psnr:.4f} dB")
    else:
        print(f"Starting {model_name} from scratch.")

    for epoch in range(start_epoch, max_epochs):
        model.train()
        train_loss = 0.0

        for noisy, clean, _ in train_loader:
            noisy = noisy.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            predicted_noise = model(noisy)
            denoised = noisy - predicted_noise
            loss = criterion(denoised, clean)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_psnr = 0.0
        with torch.no_grad():
            for noisy, clean, _ in val_loader:
                noisy = noisy.to(DEVICE, non_blocking=True)
                clean = clean.to(DEVICE, non_blocking=True)
                predicted_noise = model(noisy)
                denoised = torch.clamp(noisy - predicted_noise, 0.0, 1.0)
                val_psnr += psnr_torch(clean, denoised)

        val_psnr /= len(val_loader)
        scheduler.step(val_psnr)

        print(
            f"Epoch {epoch + 1:03d}/{max_epochs} | "
            f"train loss: {train_loss:.6f} | val PSNR: {val_psnr:.4f} dB"
        )

        if val_psnr > best_psnr:
            best_psnr = val_psnr
            patience_counter = 0
            save_model_weights(model, best_path)
            print(f"  Saved new best model to {best_path}.")
        else:
            patience_counter += 1
            print(f"  No improvement. Patience {patience_counter}/{patience}.")
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}.")
                break

        torch.save(
            {
                "epoch": epoch,
                "model_state": unwrap_model(model).state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_psnr": best_psnr,
                "patience_counter": patience_counter,
            },
            ckpt_path,
        )

    load_model_weights(model, best_path)
    print(f"Training complete for {model_name}. Best validation PSNR: {best_psnr:.4f} dB")
    return model


def evaluate_model(
    model: nn.Module,
    test_paths: Sequence[Path],
    test_labels: Sequence[int],
    model_type: str,
    model_name: str,
    sigmas: Sequence[int] = EVAL_SIGMAS,
    batch_size: int = 64,
) -> Dict[int, Dict[str, float]]:
    model.eval()
    results: Dict[int, Dict[str, float]] = {}

    for sigma in sigmas:
        dataset = ModelADataset(test_paths, test_labels, fixed_sigma=sigma) if model_type == "A" else ModelBDataset(test_paths, test_labels, fixed_sigma=sigma)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        total_psnr = 0.0
        total_ssim = 0.0

        with torch.no_grad():
            for noisy, clean, _ in loader:
                noisy = noisy.to(DEVICE, non_blocking=True)
                clean = clean.to(DEVICE, non_blocking=True)
                predicted_noise = model(noisy)
                denoised = torch.clamp(noisy - predicted_noise, 0.0, 1.0)
                total_psnr += psnr_torch(clean, denoised)
                total_ssim += ssim_torch(clean, denoised)

        results[sigma] = {
            "PSNR": total_psnr / len(loader),
            "SSIM": total_ssim / len(loader),
        }
        print(f"{model_name} | sigma={sigma:>2} | PSNR={results[sigma]['PSNR']:.4f} dB | SSIM={results[sigma]['SSIM']:.4f}")

    return results


## 9. Prepare data split


In [ ]:
image_paths, labels, classes = load_image_paths(DATASET_PATH)
train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = split_dataset(image_paths, labels)

print(f"Dataset: {DATASET_PATH}")
print(f"Classes ({len(classes)}): {classes}")
print(f"Images: {len(image_paths):,}")
print(f"Train: {len(train_paths):,} | Validation: {len(val_paths):,} | Test: {len(test_paths):,}")
print(f"Supplemental water scenes included: {INCLUDE_SUPPLEMENTAL_WATER_SCENES}")


## 10. Train Model A and Model B


In [ ]:
# Model A: 13-channel denoising
model_a = maybe_parallel(build_dncnn(num_channels=13)).to(DEVICE)

if TRAIN_MODEL_A:
    train_loader_a = DataLoader(
        ModelADataset(train_paths, train_labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader_a = DataLoader(
        ModelADataset(val_paths, val_labels, fixed_sigma=VALIDATION_SIGMA),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    model_a = train_model(model_a, train_loader_a, val_loader_a, model_name="ModelA")
else:
    path_a = model_path("ModelA")
    if not path_a.exists():
        raise FileNotFoundError(f"Model A weights not found: {path_a}")
    model_a = load_model_weights(model_a, path_a)
    print(f"Loaded Model A weights from {path_a}")

# Model B: one shared single-channel denoiser, applied to each band independently
model_b = maybe_parallel(build_dncnn(num_channels=1)).to(DEVICE)

if TRAIN_MODEL_B:
    train_loader_b = DataLoader(
        ModelBDataset(train_paths, train_labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader_b = DataLoader(
        ModelBDataset(val_paths, val_labels, fixed_sigma=VALIDATION_SIGMA),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    model_b = train_model(model_b, train_loader_b, val_loader_b, model_name="ModelB")
else:
    path_b = model_path("ModelB")
    if not path_b.exists():
        raise FileNotFoundError(f"Model B weights not found: {path_b}")
    model_b = load_model_weights(model_b, path_b)
    print(f"Loaded Model B weights from {path_b}")


## 11. Standard evaluation at selected noise levels


In [ ]:
results_a = evaluate_model(model_a, test_paths, test_labels, model_type="A", model_name="ModelA")
results_b = evaluate_model(model_b, test_paths, test_labels, model_type="B", model_name="ModelB")

print("\nResults summary")
print(f"{'Model':<10} {'Sigma':<8} {'PSNR':<10} {'SSIM':<10}")
print("-" * 42)
for sigma in EVAL_SIGMAS:
    print(f"{'ModelA':<10} {sigma:<8} {results_a[sigma]['PSNR']:<10.4f} {results_a[sigma]['SSIM']:<10.4f}")
    print(f"{'ModelB':<10} {sigma:<8} {results_b[sigma]['PSNR']:<10.4f} {results_b[sigma]['SSIM']:<10.4f}")


## 12. Detailed evaluation helpers


In [ ]:
def denoise_model_a(model: nn.Module, clean_image: np.ndarray, sigma: int) -> Tuple[np.ndarray, np.ndarray]:
    noisy, _ = add_noise(clean_image, fixed_sigma=sigma)
    noisy_tensor = torch.from_numpy(noisy).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        predicted_noise = model(noisy_tensor)
        denoised = torch.clamp(noisy_tensor - predicted_noise, 0.0, 1.0)

    return noisy, denoised.squeeze(0).cpu().numpy()


def denoise_model_b(model: nn.Module, clean_image: np.ndarray, sigma: int) -> Tuple[np.ndarray, np.ndarray]:
    noisy, _ = add_noise(clean_image, fixed_sigma=sigma)
    denoised_channels = []

    model.eval()
    with torch.no_grad():
        for channel_idx in range(clean_image.shape[0]):
            channel = noisy[channel_idx : channel_idx + 1]
            channel_tensor = torch.from_numpy(channel).unsqueeze(0).to(DEVICE)
            predicted_noise = model(channel_tensor)
            denoised_channel = torch.clamp(channel_tensor - predicted_noise, 0.0, 1.0)
            denoised_channels.append(denoised_channel.squeeze(0).cpu().numpy())

    return noisy, np.concatenate(denoised_channels, axis=0)


def run_full_sweep(test_paths: Sequence[Path], sigmas: Sequence[int] = SWEEP_SIGMAS) -> Dict[int, Dict[str, Dict[str, List[float]]]]:
    results = {
        sigma: {
            "noisy": {"PSNR": [], "SSIM": []},
            "A": {"PSNR": [], "SSIM": []},
            "B": {"PSNR": [], "SSIM": []},
        }
        for sigma in sigmas
    }

    total = len(test_paths) * len(sigmas)
    done = 0

    for sigma in sigmas:
        for path in test_paths:
            clean = load_image(path)

            np.random.seed(SEED)
            noisy, _ = add_noise(clean, fixed_sigma=sigma)
            results[sigma]["noisy"]["PSNR"].append(psnr_numpy(clean, noisy))
            results[sigma]["noisy"]["SSIM"].append(ssim_numpy(clean, noisy))

            np.random.seed(SEED)
            _, denoised_a = denoise_model_a(model_a, clean, sigma)
            results[sigma]["A"]["PSNR"].append(psnr_numpy(clean, denoised_a))
            results[sigma]["A"]["SSIM"].append(ssim_numpy(clean, denoised_a))

            np.random.seed(SEED)
            _, denoised_b = denoise_model_b(model_b, clean, sigma)
            results[sigma]["B"]["PSNR"].append(psnr_numpy(clean, denoised_b))
            results[sigma]["B"]["SSIM"].append(ssim_numpy(clean, denoised_b))

            done += 1
            if done % 500 == 0:
                print(f"Progress: {done}/{total}", end="\r")

    print("\nSweep complete.")
    print(f"\n{'Sigma':<8} {'Noisy PSNR':<14} {'A PSNR':<18} {'B PSNR':<18} {'A SSIM':<10} {'B SSIM':<10}")
    print("-" * 86)
    for sigma in sigmas:
        noisy_psnr = np.mean(results[sigma]["noisy"]["PSNR"])
        a_psnr = np.mean(results[sigma]["A"]["PSNR"])
        b_psnr = np.mean(results[sigma]["B"]["PSNR"])
        a_std = np.std(results[sigma]["A"]["PSNR"])
        b_std = np.std(results[sigma]["B"]["PSNR"])
        a_ssim = np.mean(results[sigma]["A"]["SSIM"])
        b_ssim = np.mean(results[sigma]["B"]["SSIM"])
        print(f"{sigma:<8} {noisy_psnr:<14.3f} {a_psnr:.3f} ± {a_std:.3f}     {b_psnr:.3f} ± {b_std:.3f}     {a_ssim:<10.4f} {b_ssim:<10.4f}")

    return results


def run_per_class(
    test_paths: Sequence[Path],
    test_labels: Sequence[int],
    classes: Sequence[str],
    sigmas: Sequence[int] = (25, 50),
) -> Dict[int, Dict[int, Dict[str, List[float]]]]:
    class_results = {sigma: {i: {"A": [], "B": []} for i in range(len(classes))} for sigma in sigmas}

    for sigma in sigmas:
        for path, label in zip(test_paths, test_labels):
            clean = load_image(path)

            np.random.seed(SEED)
            _, denoised_a = denoise_model_a(model_a, clean, sigma)
            class_results[sigma][label]["A"].append(psnr_numpy(clean, denoised_a))

            np.random.seed(SEED)
            _, denoised_b = denoise_model_b(model_b, clean, sigma)
            class_results[sigma][label]["B"].append(psnr_numpy(clean, denoised_b))

    for sigma in sigmas:
        print(f"\nPer-class PSNR at sigma={sigma}")
        print(f"{'Class':<25} {'Model A':<14} {'Model B':<14} {'Std A':<10} {'Std B':<10} {'Winner'}")
        print("-" * 86)
        for class_idx, class_name in enumerate(classes):
            a_scores = class_results[sigma][class_idx]["A"]
            b_scores = class_results[sigma][class_idx]["B"]
            a_mean, a_std = np.mean(a_scores), np.std(a_scores)
            b_mean, b_std = np.mean(b_scores), np.std(b_scores)
            winner = "Model A" if a_mean > b_mean else "Model B"
            print(f"{class_name:<25} {a_mean:<14.3f} {b_mean:<14.3f} {a_std:<10.3f} {b_std:<10.3f} {winner}")

    return class_results


def run_per_channel(
    test_paths: Sequence[Path],
    sigmas: Sequence[int] = EVAL_SIGMAS,
    num_samples: int = PER_CHANNEL_SAMPLES,
) -> Dict[int, Dict[int, List[float]]]:
    sampled_paths = list(test_paths)[:num_samples]
    channel_results = {sigma: {channel: [] for channel in range(13)} for sigma in sigmas}

    model_b.eval()
    with torch.no_grad():
        for sigma in sigmas:
            for path in sampled_paths:
                clean = load_image(path)
                noisy, _ = add_noise(clean, fixed_sigma=sigma)

                for channel_idx in range(13):
                    channel = noisy[channel_idx : channel_idx + 1]
                    channel_tensor = torch.from_numpy(channel).unsqueeze(0).to(DEVICE)
                    predicted_noise = model_b(channel_tensor)
                    denoised_channel = torch.clamp(channel_tensor - predicted_noise, 0.0, 1.0)
                    denoised_channel_np = denoised_channel.squeeze(0).cpu().numpy()
                    channel_results[sigma][channel_idx].append(psnr_numpy(clean[channel_idx : channel_idx + 1], denoised_channel_np))

    print("\nPer-channel PSNR for Model B")
    print(f"{'Channel':<10}", end="")
    for sigma in sigmas:
        print(f"{'sigma=' + str(sigma):<16}", end="")
    print()
    print("-" * (10 + 16 * len(sigmas)))

    for channel_idx in range(13):
        print(f"{'Band ' + str(channel_idx + 1):<10}", end="")
        for sigma in sigmas:
            mean = np.mean(channel_results[sigma][channel_idx])
            std = np.std(channel_results[sigma][channel_idx])
            print(f"{mean:.3f} ± {std:.3f}  ", end="")
        print()

    return channel_results


def run_statistical_tests(sweep_results: Dict[int, Dict[str, Dict[str, List[float]]]], sigmas: Sequence[int] = EVAL_SIGMAS) -> None:
    print("\nPaired t-test: Model A vs Model B PSNR")
    print(f"{'Sigma':<8} {'A mean':<10} {'B mean':<10} {'t-stat':<10} {'p-value':<12} {'Significant?'}")
    print("-" * 70)

    for sigma in sigmas:
        a_scores = sweep_results[sigma]["A"]["PSNR"]
        b_scores = sweep_results[sigma]["B"]["PSNR"]
        t_stat, p_value = stats.ttest_rel(a_scores, b_scores)
        significant = "YES (p<0.05)" if p_value < 0.05 else "NO"
        print(f"{sigma:<8} {np.mean(a_scores):<10.3f} {np.mean(b_scores):<10.3f} {t_stat:<10.3f} {p_value:<12.6f} {significant}")


## 13. Run detailed evaluations

The full sweep performs 82,800 image/noise-level evaluations on the supplied corpus and can take a long time. Set `RUN_DETAILED_EVALUATION = True` in the configuration cell to reproduce it. The committed report and figures contain the completed run.


In [ ]:
if RUN_DETAILED_EVALUATION:
    sweep_results = run_full_sweep(test_paths)
    per_class_results = run_per_class(test_paths, test_labels, classes)
    channel_results = run_per_channel(test_paths)
    run_statistical_tests(sweep_results)
else:
    print("Detailed evaluation skipped. Set RUN_DETAILED_EVALUATION = True to run it.")


## 14. Plot results


In [ ]:
def plot_all(
    sweep_results: Dict[int, Dict[str, Dict[str, List[float]]]],
    per_class_results: Dict[int, Dict[int, Dict[str, List[float]]]],
    channel_results: Dict[int, Dict[int, List[float]]],
    classes: Sequence[str],
) -> None:
    sigmas = sorted(sweep_results.keys())

    noisy_psnr = [np.mean(sweep_results[s]["noisy"]["PSNR"]) for s in sigmas]
    a_psnr = [np.mean(sweep_results[s]["A"]["PSNR"]) for s in sigmas]
    b_psnr = [np.mean(sweep_results[s]["B"]["PSNR"]) for s in sigmas]
    a_std = [np.std(sweep_results[s]["A"]["PSNR"]) for s in sigmas]
    b_std = [np.std(sweep_results[s]["B"]["PSNR"]) for s in sigmas]
    noisy_ssim = [np.mean(sweep_results[s]["noisy"]["SSIM"]) for s in sigmas]
    a_ssim = [np.mean(sweep_results[s]["A"]["SSIM"]) for s in sigmas]
    b_ssim = [np.mean(sweep_results[s]["B"]["SSIM"]) for s in sigmas]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(sigmas, noisy_psnr, "o--", label="Noisy", linewidth=2)
    ax.plot(sigmas, a_psnr, "s-", label="Model A", linewidth=2)
    ax.plot(sigmas, b_psnr, "^-", label="Model B", linewidth=2)
    ax.fill_between(sigmas, np.array(a_psnr) - np.array(a_std), np.array(a_psnr) + np.array(a_std), alpha=0.15)
    ax.fill_between(sigmas, np.array(b_psnr) - np.array(b_std), np.array(b_psnr) + np.array(b_std), alpha=0.15)
    ax.set_xlabel("Noise level (sigma)")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title("PSNR vs noise level with standard deviation bands")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plot_psnr_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(sigmas, noisy_ssim, "o--", label="Noisy", linewidth=2)
    ax.plot(sigmas, a_ssim, "s-", label="Model A", linewidth=2)
    ax.plot(sigmas, b_ssim, "^-", label="Model B", linewidth=2)
    ax.set_xlabel("Noise level (sigma)")
    ax.set_ylabel("SSIM")
    ax.set_title("SSIM vs noise level")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plot_ssim_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

    sigma = 25
    x = np.arange(len(classes))
    width = 0.35
    a_means = [np.mean(per_class_results[sigma][i]["A"]) for i in range(len(classes))]
    b_means = [np.mean(per_class_results[sigma][i]["B"]) for i in range(len(classes))]
    a_stds = [np.std(per_class_results[sigma][i]["A"]) for i in range(len(classes))]
    b_stds = [np.std(per_class_results[sigma][i]["B"]) for i in range(len(classes))]

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(x - width / 2, a_means, width, yerr=a_stds, label="Model A", capsize=4)
    ax.bar(x + width / 2, b_means, width, yerr=b_stds, label="Model B", capsize=4)
    ax.set_xlabel("Land-cover class")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title(f"Per-class PSNR at sigma={sigma}")
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=30, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plot_per_class.png", dpi=150, bbox_inches="tight")
    plt.show()

    x = np.arange(13)
    width = 0.25
    fig, ax = plt.subplots(figsize=(16, 6))
    for i, sigma in enumerate(EVAL_SIGMAS):
        means = [np.mean(channel_results[sigma][channel_idx]) for channel_idx in range(13)]
        ax.bar(x + i * width, means, width, label=f"sigma={sigma}")
    ax.set_xlabel("Sentinel-2 band")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title("Model B per-channel PSNR across noise levels")
    ax.set_xticks(x + width)
    ax.set_xticklabels([f"B{idx + 1}" for idx in range(13)])
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plot_per_channel.png", dpi=150, bbox_inches="tight")
    plt.show()

    gap = np.array(a_psnr) - np.array(b_psnr)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(sigmas, gap, width=3.0)
    ax.axhline(0, linewidth=1)
    ax.set_xlabel("Noise level (sigma)")
    ax.set_ylabel("PSNR difference: Model A - Model B (dB)")
    ax.set_title("Model A vs Model B PSNR gap across noise levels")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plot_gap.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"All plots saved to {OUTPUT_DIR}")


if RUN_DETAILED_EVALUATION:
    plot_all(sweep_results, per_class_results, channel_results, classes)


## 15. Operating notes

- `DATASET_PATH`, `MODEL_DIR`, and `OUTPUT_DIR` can be set in `.env`; use `.env.example` as the template.
- The supplied best weights are `models/best_model_a.pth` and `models/best_model_b.pth`.
- Training is disabled by default. Set the relevant `TRAIN_MODEL_*` switch to `True` to retrain; resumable epoch checkpoints are written below `MODEL_DIR`.
- Detailed evaluation is also opt-in because the complete 15-level sweep is expensive. Its existing output is preserved in `results/evaluation_results.txt` and `assets/results/`.
- `INCLUDE_SUPPLEMENTAL_WATER_SCENES = True` matches the 27,597-tile corpus used for the supplied artifacts. Set it to `False` for the 27,000 class-prefixed EuroSAT tiles, then retrain before comparing results.
